In [100]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder,RobustScaler
from sklearn.neighbors import KNeighborsClassifier,KNeighborsRegressor
from sklearn.metrics import classification_report,r2_score,f1_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV

In [101]:
df = pd.read_csv("fake_bills.csv",sep=';')
df

,is_genuine,diagonal,height_left,height_right,margin_low,margin_up,length
0,True,171.81,104.86,104.95,4.52,2.89,112.83
1,True,171.46,103.36,103.66,3.77,2.99,113.09
2,True,172.69,104.48,103.50,4.40,2.94,113.16
3,True,171.36,103.91,103.94,3.62,3.01,113.51
4,True,171.73,104.28,103.46,4.04,3.48,112.54
...,...,...,...,...,...,...,...
1495,False,171.75,104.38,104.17,4.42,3.09,111.28
1496,False,172.19,104.63,104.44,5.27,3.37,110.97
1497,False,171.80,104.01,104.12,5.51,3.36,111.95
1498,False,172.06,104.28,104.06,5.17,3.46,112.25


In [102]:
x = df.drop(columns=['is_genuine'],axis=1)
y = df['is_genuine']
x,y


(      diagonal  height_left  height_right  margin_low  margin_up  length
 0       171.81       104.86        104.95        4.52       2.89  112.83
 1       171.46       103.36        103.66        3.77       2.99  113.09
 2       172.69       104.48        103.50        4.40       2.94  113.16
 3       171.36       103.91        103.94        3.62       3.01  113.51
 4       171.73       104.28        103.46        4.04       3.48  112.54
 ...        ...          ...           ...         ...        ...     ...
 1495    171.75       104.38        104.17        4.42       3.09  111.28
 1496    172.19       104.63        104.44        5.27       3.37  110.97
 1497    171.80       104.01        104.12        5.51       3.36  111.95
 1498    172.06       104.28        104.06        5.17       3.46  112.25
 1499    171.47       104.15        103.82        4.63       3.37  112.07
 
 [1500 rows x 6 columns],
 0        True
 1        True
 2        True
 3        True
 4        True
         

In [103]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   is_genuine    1500 non-null   bool   
 1   diagonal      1500 non-null   float64
 2   height_left   1500 non-null   float64
 3   height_right  1500 non-null   float64
 4   margin_low    1463 non-null   float64
 5   margin_up     1500 non-null   float64
 6   length        1500 non-null   float64
dtypes: bool(1), float64(6)
memory usage: 71.9 KB


In [104]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.2,random_state=42)

In [105]:
xtrain

,diagonal,height_left,height_right,margin_low,margin_up,length
382,172.28,104.62,103.80,4.08,3.08,113.26
538,171.90,104.50,103.49,4.08,2.82,113.50
1493,171.63,104.33,104.61,4.88,3.35,112.16
1112,172.06,104.28,104.31,5.61,3.27,111.61
324,172.10,104.42,103.60,4.18,2.89,113.32
...,...,...,...,...,...,...
1130,171.56,104.29,104.19,5.23,3.63,112.94
1294,172.40,104.27,104.18,4.92,3.17,111.79
860,171.38,103.83,103.99,4.44,3.12,113.48
1459,171.78,104.31,103.82,6.19,3.25,111.14


In [106]:
1200**0.5

34.64101615137755

In [107]:
num_col = x.select_dtypes(include=['int64','float64']).columns
obj_col = x.select_dtypes(include=['object']).columns
num_col
obj_col

Index([], dtype='object')

In [ ]:
preprocessing = Pipeline(
    steps=[
        ('impute',SimpleImputer(strategy='median')),
        ('scalling',RobustScaler()),
        ('model',KNeighborsClassifier())
        
    ]
)
preprocessing.fit(xtrain,ytrain)

Pipeline(steps=[('impute', SimpleImputer(strategy='median')),
                ('scalling', RobustScaler()),
                ('model', KNeighborsClassifier())])

In [109]:
gridSearch_cv = GridSearchCV(
    estimator = num_preprocessing,
    param_grid = {
        "model__n_neighbors":[5,30,35,45,65],
        "model__weights":['uniform','distance'],
        "model__metric":['euclidean','manhattan'],
    },
  cv=10 ,
  scoring='f1'
)
  
gridSearch_cv.fit(xtrain,ytrain)

GridSearchCV(cv=10,
             estimator=Pipeline(steps=[('impute',
                                        SimpleImputer(strategy='median')),
                                       ('scalling', RobustScaler()),
                                       ('model', KNeighborsClassifier())]),
             param_grid={'model__metric': ['euclidean', 'manhattan'],
                         'model__n_neighbors': [5, 30, 35, 45, 65],
                         'model__weights': ['uniform', 'distance']},
             scoring='f1')

In [110]:
gridSearch_cv.fit(xtrain,ytrain)

GridSearchCV(cv=10,
             estimator=Pipeline(steps=[('impute',
                                        SimpleImputer(strategy='median')),
                                       ('scalling', RobustScaler()),
                                       ('model', KNeighborsClassifier())]),
             param_grid={'model__metric': ['euclidean', 'manhattan'],
                         'model__n_neighbors': [5, 30, 35, 45, 65],
                         'model__weights': ['uniform', 'distance']},
             scoring='f1')

In [111]:
gridSearch_cv.best_params_

{'model__metric': 'euclidean',
 'model__n_neighbors': 30,
 'model__weights': 'uniform'}

In [112]:
gridSearch_cv.best_estimator_

Pipeline(steps=[('impute', SimpleImputer(strategy='median')),
                ('scalling', RobustScaler()),
                ('model',
                 KNeighborsClassifier(metric='euclidean', n_neighbors=30))])

In [113]:
gridSearch_cv.best_score_

np.float64(0.9908270158985907)

In [114]:
sample = gridSearch_cv.best_estimator_

In [115]:
sample.fit(xtrain,ytrain)

Pipeline(steps=[('impute', SimpleImputer(strategy='median')),
                ('scalling', RobustScaler()),
                ('model',
                 KNeighborsClassifier(metric='euclidean', n_neighbors=30))])

In [118]:
y_pred = gridSearch_cv.predict(xtrain)


In [125]:
print(classification_report(ytrain,y_pred))

              precision    recall  f1-score   support

       False       0.99      0.97      0.98       390
        True       0.99      1.00      0.99       810

    accuracy                           0.99      1200
   macro avg       0.99      0.98      0.99      1200
weighted avg       0.99      0.99      0.99      1200



In [ ]:
y_predd =gridSearch_cv.predict(xtest)

In [124]:
print(classification_report(ytest,y_predd))

              precision    recall  f1-score   support

       False       1.00      0.96      0.98       110
        True       0.98      1.00      0.99       190

    accuracy                           0.99       300
   macro avg       0.99      0.98      0.99       300
weighted avg       0.99      0.99      0.99       300

